# Day 13 — Feature Engineering

## Objective

Create prediction-safe time-based, lag, rolling, and cross-sensor features
for the underwater water-quality time-series dataset.

### Deliverables

- Time-based features
- Lag features
- Rolling statistical features
- Cross-sensor features
- Prediction-time availability checks
- Raw vs engineered feature comparison
- Feature definitions and documentation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed"

train_df = pd.read_csv(base_path + r"\train_v1.csv")
val_df = pd.read_csv(base_path + r"\validation_v1.csv")
test_df = pd.read_csv(base_path + r"\test_v1.csv")

for data in [train_df, val_df, test_df]:
    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        errors="coerce"
    )

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (2291, 11)
Validation: (491, 11)
Test: (491, 11)


In [2]:
print("Columns:")
print(train_df.columns.tolist())

Columns:
['timestamp', 'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']


In [3]:
sensor_cols = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity"
]

target_col = "Potability"

print("Sensor features:")
print(sensor_cols)

print("\nTarget:")
print(target_col)

Sensor features:
['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

Target:
Potability


In [4]:
raw_features = sensor_cols.copy()

print("Number of raw features:", len(raw_features))
print(raw_features)

Number of raw features: 9
['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']


In [5]:
# Create time-based features

for data in [train_df, val_df, test_df]:
    data["hour"] = data["timestamp"].dt.hour
    data["day_of_week"] = data["timestamp"].dt.dayofweek
    data["day"] = data["timestamp"].dt.day
    data["month"] = data["timestamp"].dt.month

print(train_df[
    ["timestamp", "hour", "day_of_week", "day", "month"]
].head())

            timestamp  hour  day_of_week  day  month
0 2025-01-01 01:00:00     1            2    1      1
1 2025-01-01 02:00:00     2            2    1      1
2 2025-01-01 03:00:00     3            2    1      1
3 2025-01-01 04:00:00     4            2    1      1
4 2025-01-01 05:00:00     5            2    1      1


In [6]:
print("Time-based features created:")
print([
    "hour",
    "day_of_week",
    "day",
    "month"
])

Time-based features created:
['hour', 'day_of_week', 'day', 'month']


In [8]:
# Sort each dataset chronologically

for data in [train_df, val_df, test_df]:
    data.sort_values("timestamp", inplace=True)
    data.reset_index(drop=True, inplace=True)

print("Data sorted chronologically.")

Data sorted chronologically.


In [9]:
# Create lag-1 features using the previous observation

for data in [train_df, val_df, test_df]:
    for col in sensor_cols:
        data[f"{col}_lag_1"] = data[col].shift(1)

print("Lag-1 features created.")

print(train_df[
    ["timestamp"] + [f"{col}_lag_1" for col in sensor_cols]
].head())

Lag-1 features created.
            timestamp  ph_lag_1  Hardness_lag_1  Solids_lag_1  \
0 2025-01-01 01:00:00       NaN             NaN           NaN   
1 2025-01-01 02:00:00 -2.227286       -2.098750     -0.391417   
2 2025-01-01 03:00:00  0.662715        0.856166     -0.244684   
3 2025-01-01 04:00:00  0.806219        0.548784     -0.002835   
4 2025-01-01 05:00:00  1.317524       -0.488155     -0.466083   

   Chloramines_lag_1  Sulfate_lag_1  Conductivity_lag_1  Organic_carbon_lag_1  \
0                NaN            NaN                 NaN                   NaN   
1          -0.296555       0.839420            2.073831              0.274258   
2           1.363504       0.839420           -0.095518              0.779840   
3           0.598709       0.564777           -0.784362              1.249273   
4          -0.352283      -0.539208           -0.346902             -0.810106   

   Trihalomethanes_lag_1  Turbidity_lag_1  
0                    NaN              NaN  
1         

In [10]:
# Create 3-observation rolling mean features
# shift(1) ensures only past observations are used

for data in [train_df, val_df, test_df]:
    for col in sensor_cols:
        data[f"{col}_rolling_mean_3"] = (
            data[col].shift(1).rolling(window=3).mean()
        )

print("Rolling mean features created.")

print(train_df[
    ["timestamp"] + [f"{col}_rolling_mean_3" for col in sensor_cols]
].head(6))

Rolling mean features created.
            timestamp  ph_rolling_mean_3  Hardness_rolling_mean_3  \
0 2025-01-01 01:00:00                NaN                      NaN   
1 2025-01-01 02:00:00                NaN                      NaN   
2 2025-01-01 03:00:00                NaN                      NaN   
3 2025-01-01 04:00:00          -0.252784                -0.231266   
4 2025-01-01 05:00:00           0.928819                 0.305599   
5 2025-01-01 06:00:00           0.376048                -0.067588   

   Solids_rolling_mean_3  Chloramines_rolling_mean_3  Sulfate_rolling_mean_3  \
0                    NaN                         NaN                     NaN   
1                    NaN                         NaN                     NaN   
2                    NaN                         NaN                     NaN   
3              -0.212979                    0.555219                0.747872   
4              -0.237867                    0.536643                0.288330   
5    

In [11]:
# Create 3-observation rolling standard deviation features
# shift(1) ensures only past observations are used

for data in [train_df, val_df, test_df]:
    for col in sensor_cols:
        data[f"{col}_rolling_std_3"] = (
            data[col].shift(1).rolling(window=3).std()
        )

print("Rolling standard deviation features created.")

print(train_df[
    ["timestamp"] + [f"{col}_rolling_std_3" for col in sensor_cols]
].head(6))

Rolling standard deviation features created.
            timestamp  ph_rolling_std_3  Hardness_rolling_std_3  \
0 2025-01-01 01:00:00               NaN                     NaN   
1 2025-01-01 02:00:00               NaN                     NaN   
2 2025-01-01 03:00:00               NaN                     NaN   
3 2025-01-01 04:00:00          1.711474                1.624574   
4 2025-01-01 05:00:00          0.344190                0.704382   
5 2025-01-01 06:00:00          1.215080                0.545496   

   Solids_rolling_std_3  Chloramines_rolling_std_3  Sulfate_rolling_std_3  \
0                   NaN                        NaN                    NaN   
1                   NaN                        NaN                    NaN   
2                   NaN                        NaN                    NaN   
3              0.196222                   0.830883               0.158565   
4              0.231699                   0.859575               0.729706   
5              0.623935

# Create cross-sensor interaction features

for data in [train_df, val_df, test_df]:
    data["solids_conductivity_ratio"] = (
        data["Solids"] / data["Conductivity"].replace(0, np.nan)
    )

    data["solids_conductivity_product"] = (
        data["Solids"] * data["Conductivity"]
    )

print("Cross-sensor features created.")

print(train_df[
    [
        "timestamp",
        "Solids",
        "Conductivity",
        "solids_conductivity_ratio",
        "solids_conductivity_product"
    ]
].head())

In [15]:
# Check engineered features and prediction-time availability

print("Total columns:", len(train_df.columns))

print("\nAll columns:")
print(train_df.columns.tolist())

print("\nTarget column:")
print(target_col)

print("\nNumber of missing values in engineered features:")
print(train_df.isna().sum().sort_values(ascending=False).head(15))

Total columns: 42

All columns:
['timestamp', 'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability', 'hour', 'day_of_week', 'day', 'month', 'ph_lag_1', 'Hardness_lag_1', 'Solids_lag_1', 'Chloramines_lag_1', 'Sulfate_lag_1', 'Conductivity_lag_1', 'Organic_carbon_lag_1', 'Trihalomethanes_lag_1', 'Turbidity_lag_1', 'ph_rolling_mean_3', 'Hardness_rolling_mean_3', 'Solids_rolling_mean_3', 'Chloramines_rolling_mean_3', 'Sulfate_rolling_mean_3', 'Conductivity_rolling_mean_3', 'Organic_carbon_rolling_mean_3', 'Trihalomethanes_rolling_mean_3', 'Turbidity_rolling_mean_3', 'ph_rolling_std_3', 'Hardness_rolling_std_3', 'Solids_rolling_std_3', 'Chloramines_rolling_std_3', 'Sulfate_rolling_std_3', 'Conductivity_rolling_std_3', 'Organic_carbon_rolling_std_3', 'Trihalomethanes_rolling_std_3', 'Turbidity_rolling_std_3']

Target column:
Potability

Number of missing values in engineered features:
Turbidity_rolling_mean_3     

In [16]:
# Compare raw features with engineered features

time_features = [
    "hour",
    "day_of_week",
    "day",
    "month"
]

lag_features = [
    f"{col}_lag_1" for col in sensor_cols
]

rolling_mean_features = [
    f"{col}_rolling_mean_3" for col in sensor_cols
]

rolling_std_features = [
    f"{col}_rolling_std_3" for col in sensor_cols
]

cross_sensor_features = [
    "solids_conductivity_ratio",
    "solids_conductivity_product"
]

engineered_features = (
    time_features
    + lag_features
    + rolling_mean_features
    + rolling_std_features
    + cross_sensor_features
)

print("Raw features:", len(raw_features))
print("Time-based features:", len(time_features))
print("Lag features:", len(lag_features))
print("Rolling mean features:", len(rolling_mean_features))
print("Rolling std features:", len(rolling_std_features))
print("Cross-sensor features:", len(cross_sensor_features))
print("Total engineered features:", len(engineered_features))
print("Total features including raw:", len(raw_features) + len(engineered_features))

Raw features: 9
Time-based features: 4
Lag features: 9
Rolling mean features: 9
Rolling std features: 9
Cross-sensor features: 2
Total engineered features: 33
Total features including raw: 42


In [17]:
# Feature definition table

feature_definitions = pd.DataFrame({
    "Feature Group": [
        "Raw",
        "Time-based",
        "Lag",
        "Rolling Mean",
        "Rolling Std",
        "Cross-sensor"
    ],
    "Features": [
        ", ".join(raw_features),
        ", ".join(time_features),
        "All sensor lag-1 features",
        "All sensor rolling mean (window=3) features",
        "All sensor rolling std (window=3) features",
        "solids_conductivity_ratio, solids_conductivity_product"
    ],
    "Count": [
        len(raw_features),
        len(time_features),
        len(lag_features),
        len(rolling_mean_features),
        len(rolling_std_features),
        len(cross_sensor_features)
    ],
    "Prediction-Time Availability": [
        "Yes",
        "Yes",
        "Yes - past observation",
        "Yes - past 3 observations",
        "Yes - past 3 observations",
        "Yes - current sensor values"
    ]
})

feature_definitions

,Feature Group,Features,Count,Prediction-Time Availability
0,Raw,"ph, Hardness, Solids, Chloramines, Sulfate, Co...",9,Yes
1,Time-based,"hour, day_of_week, day, month",4,Yes
2,Lag,All sensor lag-1 features,9,Yes - past observation
3,Rolling Mean,All sensor rolling mean (window=3) features,9,Yes - past 3 observations
4,Rolling Std,All sensor rolling std (window=3) features,9,Yes - past 3 observations
5,Cross-sensor,"solids_conductivity_ratio, solids_conductivity...",2,Yes - current sensor values


In [18]:
# Save engineered datasets

output_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed"

train_df.to_csv(output_path + r"\train_v1_engineered.csv", index=False)
val_df.to_csv(output_path + r"\validation_v1_engineered.csv", index=False)
test_df.to_csv(output_path + r"\test_v1_engineered.csv", index=False)

print("Engineered datasets saved successfully.")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Engineered datasets saved successfully.
Train: (2291, 42)
Validation: (491, 42)
Test: (491, 42)


In [ ]:
# Check engineered features and prediction-time availability

print("Total columns:", len(train_df.columns))

print("\nAll columns:")
print(train_df.columns.tolist())

print("\nTarget column:")
print(target_col)

print("\nNumber of missing values in engineered features:")
print(train_df.isna().sum().sort_values(ascending=False).head(15))

Total columns: 42

All columns:
['timestamp', 'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability', 'hour', 'day_of_week', 'day', 'month', 'ph_lag_1', 'Hardness_lag_1', 'Solids_lag_1', 'Chloramines_lag_1', 'Sulfate_lag_1', 'Conductivity_lag_1', 'Organic_carbon_lag_1', 'Trihalomethanes_lag_1', 'Turbidity_lag_1', 'ph_rolling_mean_3', 'Hardness_rolling_mean_3', 'Solids_rolling_mean_3', 'Chloramines_rolling_mean_3', 'Sulfate_rolling_mean_3', 'Conductivity_rolling_mean_3', 'Organic_carbon_rolling_mean_3', 'Trihalomethanes_rolling_mean_3', 'Turbidity_rolling_mean_3', 'ph_rolling_std_3', 'Hardness_rolling_std_3', 'Solids_rolling_std_3', 'Chloramines_rolling_std_3', 'Sulfate_rolling_std_3', 'Conductivity_rolling_std_3', 'Organic_carbon_rolling_std_3', 'Trihalomethanes_rolling_std_3', 'Turbidity_rolling_std_3']

Target column:
Potability

Number of missing values in engineered features:
Turbidity_rolling_mean_3     